# Model data preparation
Model goal: classify building footprints as houses or not

Inputs:
- area
- type of nearest road
- distance to nearest road
- no. of buildings within 50 meters (100 meters)
- no. of POIs within 50 meters (100 meters)
- population density of city

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

In [2]:
# OSM buildings as a stand-in for Google OpenBuildings
bldg = gpd.read_file("..//data//ncr//OSM.gpkg", layer="gis_osm_buildings_a_free_1").to_crs(crs="EPSG:32651")
bldg["area"] = bldg.area
bldg['centroid'] = bldg.geometry.centroid

bldg_wtype = bldg.set_geometry("centroid").loc[~pd.isna(bldg["type"])].reset_index(drop=True)

In [3]:
bldg2_temp = gpd.read_file("..//data//ncr//buildings.gpkg").to_crs(crs="EPSG:32651")
bldg2_temp["area"] = bldg2_temp.area
bldg2_temp['centroid'] = bldg2_temp.geometry.centroid

bldg2 = bldg2_temp.set_geometry("centroid")

In [4]:
poi_points = gpd.read_file("..//data//ncr//OSM.gpkg", layer="gis_osm_pois_free_1").to_crs(crs="EPSG:32651")
poi_poly = gpd.read_file("..//data//ncr//OSM.gpkg", layer="gis_osm_pois_a_free_1").to_crs(crs="EPSG:32651")
poi_poly['centroid'] = poi_poly.geometry.centroid
poi_cent = poi_poly.set_geometry("centroid").drop(columns=["geometry"])
poi_cent["geometry"] = poi_cent.geometry

# Combine POI points data and POI polygon data (turned to points) into one geodataframe
poi_all = pd.concat([poi_points,poi_cent])

# Get type of and distance to nearest road
road = gpd.read_file("..//data//ncr//OSM.gpkg", layer="gis_osm_roads_free_1").to_crs(crs="EPSG:32651")
road = road.loc[~road["fclass"].isin(["footway","cycleway"])]
bldg_wtype = bldg_wtype.sjoin_nearest(road[["fclass","geometry"]],how="left",distance_col="dist_road",lsuffix=None,rsuffix="road").drop(columns=["index_road"])

# Get id of enclosing (0.03 deg x 0.03 deg) block for test-train split later
block = gpd.read_file("..//data//ncr//blocks.gpkg").to_crs(crs="EPSG:32651").reset_index(names="block_id")[["geometry","block_id"]]
bldg_wtype = bldg_wtype.sjoin(block,how="left",predicate="within",rsuffix=None).drop(columns=["index_None"])

# Get population density of the associated city
city = gpd.read_file("..//data//ncr//adm_bounds.gpkg",layer="city").to_crs(crs="EPSG:32651")
city["density"] = (1000**2*city["pop"]/city.area).astype(int)
bldg_wtype = bldg_wtype.sjoin(city[["geometry","density"]],how="left",predicate="within",rsuffix=None)

del road, block, city

In [5]:
bldg_wtype_coords = np.array(list(bldg_wtype.geometry.apply(lambda x: (x.x, x.y))))
bldg_coords = np.array(list(bldg2.geometry.apply(lambda x: (x.x, x.y))))
poi_coords = np.array(list(poi_all.geometry.apply(lambda x: (x.x, x.y))))

btree_bldg = cKDTree(bldg_coords)
btree_poi = cKDTree(poi_coords)

# Get buildings (and POIs) within 100 meters
temp = [len(x) for x in btree_bldg.query_ball_point(bldg_wtype_coords[:len(bldg_wtype_coords)//2], 100)]
bldg_wtype["within100m_bldg"] = temp + [len(x) for x in btree_bldg.query_ball_point(bldg_wtype_coords[len(bldg_wtype_coords)//2:], 100)]
temp = [len(x) for x in btree_poi.query_ball_point(bldg_wtype_coords[:len(bldg_wtype_coords)//2], 100)]
bldg_wtype["within100m_poi"] = temp + [len(x) for x in btree_poi.query_ball_point(bldg_wtype_coords[len(bldg_wtype_coords)//2:], 100)]

# Get buildings (and POIs) within 50 meters
temp = [len(x) for x in btree_bldg.query_ball_point(bldg_wtype_coords[:len(bldg_wtype_coords)//2], 50)]
bldg_wtype["within50m_bldg"] = temp + [len(x) for x in btree_bldg.query_ball_point(bldg_wtype_coords[len(bldg_wtype_coords)//2:], 50)]
temp = [len(x) for x in btree_poi.query_ball_point(bldg_wtype_coords[:len(bldg_wtype_coords)//2], 50)]
bldg_wtype["within50m_poi"] = temp + [len(x) for x in btree_poi.query_ball_point(bldg_wtype_coords[len(bldg_wtype_coords)//2:], 50)]

In [6]:
# Save model training dataset
bldg_wtype[["name","type","area","fclass_road","dist_road","within100m_bldg","within100m_poi","within50m_bldg","within50m_poi","block_id","density"]].to_csv("..//data//modeling//dataset.csv",index=False)